# GeoIlm — Navoiy TTS bilan ovoz kutubxonasini yasash (Google Colab, bepul GPU)

**Bir marta** ishga tushiriladi. Natija: `voice_lines.zip` → ichida har bir ibora uchun `.wav` fayl.
Uni loyihadagi `quiz_generator/voice/lines/` papkasiga ochasiz.

**Avval:** `Runtime → Change runtime type → T4 GPU` ni tanlang.

Kerakli 2 ta fayl (keyingi qadamda yuklaysiz):
1. `reference.wav` — **o'z ovozingiz**, 6–12 soniya, shovqinsiz, bir tekis gapirilgan (masalan: *"Assalomu alaykum, GeoIlm kanaliga xush kelibsiz. Bugun yana yangi viktorina."*). Model shu ovoz tembrida gapiradi.
2. `voice_lines.json` — loyihadagi `data/voice_lines.json` (`python tools/voice_lines.py` yasaydi).

Model: [aisha-org/navoiy-tts](https://huggingface.co/aisha-org/navoiy-tts) (Apache-2.0), asos: CosyVoice2-0.5B.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. O'rnatish (5–10 daqiqa)

In [ ]:
%cd /content
!git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git
%cd /content/CosyVoice
!git checkout 074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc
!git submodule update --init --recursive
!pip install -q -r requirements.txt
!pip install -q huggingface_hub==0.36.2 transformers==4.51.3
%cd /content

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('FunAudioLLM/CosyVoice2-0.5B', local_dir='/content/CosyVoice/pretrained_models/CosyVoice2-0.5B')
snapshot_download('aisha-org/navoiy-tts', local_dir='/content/navoiy-tts')
print('Yuklab olindi')

## 2. `reference.wav` va `voice_lines.json` ni yuklang

In [ ]:
from google.colab import files
up = files.upload()   # ikkala faylni birga tanlang
print(list(up))

## 3. Modelni yuklash (bir marta)

In [ ]:
import sys, json, random, torch, torchaudio
sys.path.insert(0, '/content/CosyVoice')
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')
sys.path.insert(0, '/content/navoiy-tts')
from cosyvoice.cli.cosyvoice import CosyVoice2
from uztts.normalize import normalize

def unwrap(v):
    if not isinstance(v, dict): return v
    for k in ('state_dict', 'model', 'llm'):
        if isinstance(v.get(k), dict): return unwrap(v[k])
    return v

model = CosyVoice2('/content/CosyVoice/pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=True)
state = unwrap(torch.load('/content/navoiy-tts/emotion_600h_joint.pt', map_location='cpu', weights_only=True))
r = model.model.llm.load_state_dict(state, strict=False)
print('missing:', len(r.missing_keys), 'unexpected:', len(r.unexpected_keys))
model.model.llm.eval()

emos = json.load(open('/content/navoiy-tts/emotions_40h.json', encoding='utf-8'))
lookup = {}
for e in emos:
    lookup[e['uz'].lower()] = e
    for t in e['tag'].replace('[', ' ').replace(']', ' ').split():
        lookup[t.lower()] = e
print('Uslublar:', sorted(lookup))

## 4. Bitta sinov — ovozni tinglab ko'ring
Yoqmasa: `EMOTION` ni o'zgartiring (`happy`, `warm`, `calm`, `surprised`) yoki boshqa `reference.wav` yozing.

In [ ]:
EMOTION = 'happy'
SPEED = 1.05
REF = '/content/reference.wav'

def say(text, out, emotion=EMOTION, seed=1986):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    instr = lookup[emotion]['instruct'].strip() + '<|endofprompt|>'
    chunks = []
    with torch.inference_mode():
        for res in model.inference_instruct2(normalize(text, mode='infer'), instr, REF, stream=False, speed=SPEED):
            chunks.append(res['tts_speech'].detach().cpu())
    audio = torch.cat(chunks, dim=1)
    torchaudio.save(out, audio, 24000)
    return audio.shape[-1] / 24000

from IPython.display import Audio, display
print(say('Bu qaysi davlat?', '/content/test.wav'), 's')
display(Audio('/content/test.wav'))

## 5. Hammasini o'qitish (~150–400 ibora, 10–30 daqiqa)

In [ ]:
import os, zipfile
lines = json.load(open('/content/voice_lines.json', encoding='utf-8'))
os.makedirs('/content/lines', exist_ok=True)
bad = []
for i, (key, text) in enumerate(lines.items(), 1):
    out = f'/content/lines/{key}.wav'
    if os.path.exists(out): continue
    try:
        d = say(text, out, emotion='happy' if key.startswith(('title_', 'outro')) else EMOTION)
        if d > 6: bad.append((key, text, round(d, 1)))   # juda uzun — model takrorlagan bo'lishi mumkin
    except Exception as e:
        bad.append((key, text, str(e)))
    if i % 20 == 0: print(f'{i}/{len(lines)}')
print('Tayyor. Shubhali:', bad)

In [ ]:
with zipfile.ZipFile('/content/voice_lines.zip', 'w') as z:
    for f in os.listdir('/content/lines'):
        z.write(f'/content/lines/{f}', f)
files.download('/content/voice_lines.zip')

## Keyin
1. `voice_lines.zip` ni loyihadagi `quiz_generator/voice/lines/` papkasiga oching.
2. Yangi davlat/futbolchi qo'shsangiz: `python tools/voice_lines.py` → yangi `voice_lines.json` ni shu daftarga yuklab, faqat 5–6-qadamni qayta ishga tushiring (bor fayllar o'tkazib yuboriladi — buning uchun avvalgi `.wav` larni `/content/lines/` ga yuklab qo'ying).